# Gratimos · shapes

**Infer what a source actually contains, then cast into it safely — and report every value you had to bend.** A cast that silently coerced is a data-quality bug that surfaces three systems later.

> **Every cell in this notebook runs.** They are generated from
> [`tools/notebooks/spec.py`](../tools/notebooks/spec.py) and executed by CI, so a
> cell that cannot run does not reach a commit. Change a cell, re-run it, and the
> page is yours — that is what it is for.


The unit is a **`DataShape`**: a named set of `FieldShape`s, each with a type
tag, a nullability, and the observations behind it. Shapes merge, so a shape
inferred from a thousand rows and one from a different thousand union into a
shape that covers both.

Casting has two modes. **Lenient** bends what it can and reports it; **strict**
refuses. Which you want depends on whether a surprise is a nuisance or a
correctness problem — and the library will not decide that for you.

In [ ]:
# --- setup: works locally, on Binder, and on Colab -------------------------
import subprocess, sys, pathlib

def _ensure_installed():
    """Install the package if it is not importable. No-op when it already is."""
    try:
        import slpie, gratimos          # noqa: F401
        return pathlib.Path(slpie.__file__).parent.parent
    except ModuleNotFoundError:
        pass
    here = pathlib.Path.cwd()
    root = next(
        (p for p in [here, *here.parents] if (p / "pyproject.toml").exists()), None,
    )
    if root is None:                     # Colab: no checkout, so fetch one
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/Reimain/Macropol-s.git", "/content/Macropol-s"],
            check=True,
        )
        root = pathlib.Path("/content/Macropol-s")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(root)],
                   check=True)
    sys.path.insert(0, str(root))
    return root

ROOT = _ensure_installed()
print("package root:", ROOT)

import slpie
print("slpie", slpie.__version__)

## Infer a shape from records

In [ ]:
from gratimos.meta.infer import infer_shape

orders = [
    {"id": 1, "customer": "Ada",  "total": 99.50, "placed": "2026-01-14", "priority": True},
    {"id": 2, "customer": "Lin",  "total": None,  "placed": "2026-01-15", "priority": False},
    {"id": 3, "customer": "Mo",   "total": 12.00, "placed": "2026-01-16", "priority": True},
]

shape = infer_shape(orders, name="Orders")

print(f"shape: {shape.name}  ({len(shape.fields)} fields)\n")
for field in shape.fields:
    null = "nullable" if field.nullable else "required"
    print(f"  {field.name:12} {field.tag.value:10} {null:9} "
          f"nulls {field.nulls}/{field.observations}")

`total` is nullable because one row had no value — inferred from the data rather than declared, which is the difference between a schema you wrote and one you know is true.

In [ ]:
print("as Python type hints:\n")
for field in shape.fields:
    print(f"  {field.name}: {field.python_type}")

## Shapes merge

In [ ]:
more = infer_shape(
    [{"id": 4, "customer": "Zed", "total": "45.00", "placed": "2026-02-01",
      "priority": True, "notes": "gift wrap"}],
    name="Orders",
)

merged = shape.merge(more)
print(f"merged: {len(merged.fields)} fields (was {len(shape.fields)})\n")
for field in merged.fields:
    seen = f"{field.observations} observation(s)"
    print(f"  {field.name:12} {field.tag.value:10} {seen}")

`total` arrived as a string in the new batch and as a float in the old one. The merge **promotes** to a type that holds both rather than picking a winner — losing data to a type decision is the failure this is built to avoid.

In [ ]:
from gratimos.meta.shapes import promote, TypeTag

pairs = [
    (TypeTag.INT, TypeTag.FLOAT),
    (TypeTag.INT, TypeTag.STRING),
    (TypeTag.BOOL, TypeTag.INT),
    (TypeTag.FLOAT, TypeTag.DECIMAL),
]
for left, right in pairs:
    print(f"  {left.value:8} + {right.value:8} -> {promote(left, right).value}")

## Cast into a shape, and hear about every bend

In [ ]:
from gratimos.meta.cast import CastMode, Caster

caster = Caster(shape, mode=CastMode.LENIENT)

messy = {"id": "7", "customer": "Ash", "total": "88.25",
         "placed": "2026-03-02", "priority": "yes"}

record, report = caster.record(messy)

print("cast result:")
for key, value in record.items():
    print(f"  {key:12} {value!r:24} {type(value).__name__}")
print()
print("converted:", report.converted, " <- values whose type had to change")
print("unchanged:", report.unchanged)
print("defaulted:", report.defaulted)
print("failures: ", report.failures)
print()
for note in report.notes:
    print("  ·", note)

Every one of those is reported. A cast that quietly turned `"yes"` into `True` and said nothing is how a pipeline develops a belief nobody checked.

## Strict mode refuses instead

In [ ]:
from gratimos.errors import CastError

strict = Caster(shape, mode=CastMode.STRICT)
try:
    strict.record({"id": "not-a-number", "customer": "X", "total": 1.0,
                   "placed": "2026-01-01", "priority": True})
except CastError as error:
    print("refused:", error)

Note the exception type. `CastError` carries the value and the target tag, so a caller routes on the type rather than parsing the message — the rule both error taxonomies in this repository open by stating.

## Reading real sources

In [ ]:
import csv, json, pathlib, sqlite3, tempfile

WORK = pathlib.Path(tempfile.mkdtemp(prefix="gratimos-nb-"))

# CSV
csv_path = WORK / "orders.csv"
with csv_path.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["id", "customer", "total"])
    writer.writeheader()
    writer.writerows([{"id": 1, "customer": "Ada", "total": 99.5},
                      {"id": 2, "customer": "Lin", "total": 12.0}])

# JSON lines
jsonl_path = WORK / "orders.jsonl"
jsonl_path.write_text("\n".join(
    json.dumps({"id": i, "customer": name, "total": float(i * 10)})
    for i, name in enumerate(["Ada", "Lin", "Mo"], start=1)
))

# SQLite
db_path = WORK / "orders.db"
connection = sqlite3.connect(db_path)
connection.execute("CREATE TABLE orders (id INTEGER, customer TEXT, total REAL)")
connection.executemany("INSERT INTO orders VALUES (?, ?, ?)",
                       [(1, "Ada", 99.5), (2, "Lin", 12.0)])
connection.commit()
connection.close()

for path in (csv_path, jsonl_path, db_path):
    print(f"  {path.name:14} {path.stat().st_size:5} bytes")

In [ ]:
from gratimos.probes.base import Target
from gratimos.probes.registry import default_registry

probes = default_registry()
print(f"{len(probes.names())} probes registered:")
for name in probes.names():
    print("  ", name)

In [ ]:
for path in (csv_path, jsonl_path, db_path):
    target = Target(uri=path.resolve().as_uri(), path=path)
    match = probes.match(target)
    if match is None:
        print(f"  {path.name:14} no probe claims it")
        continue
    capture = probes.capture(target)
    for payload in (capture.payloads if capture else ())[:1]:
        fields = ", ".join(f"{f.name}:{f.tag.value}" for f in payload.shape.fields)
        print(f"  {path.name:14} {match.probe.name:10} -> {fields}")

One interface, three storage formats. The shape is the same idea in each case, which is what lets everything downstream stop caring where the data came from.

In [ ]:
# Scratch cell — infer a shape from your own records.
mine = [{"sku": "A-1", "qty": 3, "price": "9.99"},
        {"sku": "B-2", "qty": 1, "price": "24.00"}]
line = infer_shape(mine, name="Line")
print(line)
for field in line.fields:
    print(f"  {field.name:8} {field.tag.value:8} {field.python_type}")